# GraMM-RAG -- MP-DocVQA End-to-End Implementation

**Single-benchmark pilot notebook** for the dissertation.  
Dataset: MP-DocVQA (val split) -- 5,187 QA records, 927 unique documents, avg 37 pages/doc.  
Data already local: 64,057 Textract OCR JSON files + 64,057 page images.  
Primary metric: **ANLS** (Average Normalised Levenshtein Similarity -- DocVQA standard).

| Phase | Description |
|---|---|
| 0 | Setup & configuration |
| 1 | Load & filter data (pilot: 100 questions) |
| 1.5 | Exploratory data analysis -- corpus stats, charts, tables |
| 2 | Verify OCR + image file coverage for pilot documents |
| 3 | Parse docs with mpdocvqa_parser (Textract OCR) + temporal annotation |
| 4 | Compute embeddings (E5-Mistral text) + build PyG graphs |
| 5 | Train HGT with evidence-guided InfoNCE loss |
| 6 | Train query router (DeBERTa-v3-base) |
| 7 | Tune reward function (alpha, beta, lambda, tau grid search) |
| 8 | Flat-vector RAG baseline (FAISS + LLM) |
| 9 | GraMM-RAG full evaluation (single seed; bootstrap CI in analysis) |
| 10 | Results comparison table |

**To extend to the full run:** change `N_PILOT = None` in Phase 0 and re-run.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PHASE 0 -- Setup & Configuration
# ═══════════════════════════════════════════════════════════════════════════

# --- PILOT FILTER: change ONLY this line to extend to the full run ---------
N_PILOT = 100      # <- set None for the full 5,187-question run
# ---------------------------------------------------------------------------

SEED    = 42

HGT_EPOCHS    = 5  if N_PILOT else 50   # <- EXTEND: 50 for full run
ROUTER_EPOCHS = 1  if N_PILOT else 3    # <- EXTEND: 3 for full run
# Embeddings are ALWAYS computed (real E5). Decoupled from N_PILOT: the
# pilot-doc filter keeps it to ~100 docs on CPU, and meaningful retrieval
# (FAISS / APPA) requires real embeddings -- skipping them makes every
# downstream metric noise.
SKIP_EMBEDDING  = False
# Embed ALL pages: capping pages would hide answer pages from retrieval
# and corrupt APPA. (No longer a speed lever -- pilot-doc filter handles that.)
MAX_PAGES_PILOT = None
# Generator context cap (node mode): the LLM sees only the top-N nodes.
GEN_TOP_N = 8
# Page-aware context: instead of N scattered nodes, feed the LLM the FULL
# text (reading order) of the top-ranked page(s). DocVQA answers are a
# short span on ONE page, so a high APPA (we know the page) only helps if
# the whole page is present -- the node cap was dropping the answer cell.
# Applied to BOTH systems for a fair comparison.
PAGE_CONTEXT       = True   # False -> legacy top-N node mode
PAGE_CONTEXT_N     = 2      # top-1 + top-2 ranked pages
PAGE_CONTEXT_CHARS = 8000   # build_prompt truncation in page mode
# Page selection mode:
#   'first_seen' (DEFAULT, empirically best: GraMM ALL 0.593) -- pick pages
#       in the order their highest-ranked node appears. The answer page is
#       a sparse, precise hit; density-based aggregation favours tangential
#       pages graph expansion floods with nodes (sum regressed to 0.524).
#   'mean_topk' -- peak relevance per page (ignores density).
#   'sum'       -- total retrieved-node score per page (density-biased).
PAGE_SCORE_MODE    = 'first_seen'  # 'first_seen' | 'mean_topk' | 'sum'
PAGE_SCORE_TOPK    = 3             # k for mean_topk
# Cross-encoder reranker (research-backed precision stage). The bi-encoder
# rerank trips on close-but-wrong candidates (dates/labels) -- a cross-
# encoder jointly scores (question, node text) and fixes exactly that.
# Literature: +2.8-13 ANLS on MP-DocVQA; effective with >=20 candidates.
USE_CROSS_ENCODER  = True
CROSS_ENCODER_MODEL = 'cross-encoder/ms-marco-MiniLM-L-6-v2'  # CPU-fast
RERANK_POOL        = 20    # vector-path candidates fed to the reranker

import sys, pathlib
ROOT = pathlib.Path('.').resolve()          # flat repo root (src/ alongside notebook)
sys.path.insert(0, str(ROOT))

DATA_DIR   = ROOT / 'data'        / 'mpdocvqa'
OCR_DIR    = DATA_DIR / 'ocr'
IMG_DIR    = DATA_DIR / 'images'
IMDB_DIR   = DATA_DIR / 'imdbs'
# Derived artefacts live at the repo root (flat, self-contained layout).
PARSED_DIR  = ROOT / 'parsed'
EMB_DIR     = ROOT / 'embeddings'
GRAPH_DIR   = ROOT / 'graphs'
MODEL_DIR   = ROOT / 'results' / 'models'
RESULTS_DIR = ROOT / 'results'

# --- Fallback path variables (defined early so later phases never NameError)
hgt_save_path   = MODEL_DIR / 'hgt_mpdocvqa'    / 'best_model.pt'
reward_save     = MODEL_DIR / 'reward_mpdocvqa.json'
router_save_dir = MODEL_DIR / 'router_mpdocvqa' / 'best_model'
vector_out_path = RESULTS_DIR / 'baseline_vector_mpdocvqa_pilot.json'
faiss_indices   = {}       # populated in Phase 8
gramm_results   = {}       # populated in Phase 9
pilot_doc_ids   = []       # populated in Phase 1

print(f'ROOT: {ROOT}')
print(f'N_PILOT = {N_PILOT}  (set None for full 5,187-question run)')

In [ ]:
import json, random, re, logging, os
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import torch

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger('mpdocvqa_e2e')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Load API keys from .env (mpdocvqa_benchmark/.env, then project root .env)
try:
    from dotenv import load_dotenv
    _loaded = load_dotenv(dotenv_path=pathlib.Path('.env'), override=False)
    if not _loaded:
        load_dotenv(dotenv_path=ROOT / '.env', override=False)
    print('API keys loaded from .env')
except ImportError:
    print('python-dotenv not installed -- reading keys from shell environment only')

if not os.environ.get('TOGETHER_API_KEY') or os.environ.get('TOGETHER_API_KEY','').startswith('your-'):
    print('Warning: TOGETHER_API_KEY not set -- generation will use extractive fallback')
    os.environ.pop('TOGETHER_API_KEY', None)
if not os.environ.get('OPENAI_API_KEY') or os.environ.get('OPENAI_API_KEY','').startswith('your-'):
    print('Warning: OPENAI_API_KEY not set -- KG triplets will be empty (safe to proceed)')
    os.environ.pop('OPENAI_API_KEY', None)

In [ ]:
for d in [DATA_DIR, OCR_DIR, IMG_DIR, PARSED_DIR, EMB_DIR, GRAPH_DIR,
          MODEL_DIR / 'hgt_mpdocvqa',
          MODEL_DIR / 'router_mpdocvqa',
          RESULTS_DIR / 'figures']:
    d.mkdir(parents=True, exist_ok=True)
print('All directories ready.')

## Phase 1 -- Load & Filter Data

Load the MP-DocVQA val split from `imdbs/imdb_val.npy` (numpy format).  
Each record contains: `question`, `valid_answers`, `image_name` (page filenames),
`pages` (context window page indices), `answer_page_idx` (which page in `pages` holds the answer),
`total_doc_pages` (full document length).  

Derived fields added:
- `doc_id`: prefix of `image_name[0]` before `_pN`
- `_actual_answer_page`: `pages[answer_page_idx]` (0-indexed absolute page number)
- `_q_words`, `_a_words`: word counts for EDA

In [ ]:
imdb = np.load(str(IMDB_DIR / 'imdb_val.npy'), allow_pickle=True)
# imdb[0] is metadata header; QA records start at index 1

def extract_doc_id(image_name):
    """Strip _pN suffix from image name to get doc_id."""
    m = re.match(r'^(.+)_p(\d+)$', str(image_name))
    return m.group(1) if m else str(image_name)

all_questions = []
for rec in imdb[1:]:
    img_names  = list(rec.get('image_name', []))
    pages_list = list(rec.get('pages', []))
    ans_idx    = int(rec.get('answer_page_idx', 0))
    total_pgs  = int(rec.get('total_doc_pages', 1))

    doc_id = extract_doc_id(img_names[0]) if img_names else ''
    actual_ans_page = pages_list[ans_idx] if ans_idx < len(pages_list) else 0

    valid_ans = list(rec.get('valid_answers', rec.get('answers', [])))
    valid_ans = [str(a) for a in valid_ans if a]
    primary_ans = valid_ans[0] if valid_ans else ''

    all_questions.append({
        'question_id':        int(rec.get('question_id', 0)),
        'question':           str(rec.get('question', '')),
        'answers':            valid_ans,      # list (multi-annotator)
        'doc_id':             doc_id,
        'image_names':        img_names,
        'pages':              pages_list,
        'answer_page_idx':    ans_idx,
        'total_doc_pages':    total_pgs,
        '_actual_answer_page': actual_ans_page,
        '_q_words':           len(str(rec.get('question', '')).split()),
        '_a_words':           len(primary_ans.split()),
        '_unanswerable':      False,          # MP-DocVQA has no unanswerable Qs
        # evidence_pages for router training: actual answer page
        '_evidence_pages':    [actual_ans_page],
        '_evidence_sources':  ['text'],        # all MP-DocVQA content is text
    })

random.seed(SEED)
pilot_questions = random.sample(all_questions, N_PILOT) if N_PILOT else all_questions
pilot_doc_ids   = sorted({q['doc_id'] for q in pilot_questions})

print(f'Total QA records  : {len(all_questions):,}')
print(f'Pilot questions   : {len(pilot_questions):,}')
print(f'Unique pilot docs : {len(pilot_doc_ids)}')

In [ ]:
import pandas as pd

n_unans       = sum(q['_unanswerable'] for q in pilot_questions)
mean_q_words  = np.mean([q['_q_words'] for q in pilot_questions])
mean_a_words  = np.mean([q['_a_words'] for q in pilot_questions])
mean_pgs      = np.mean([q['total_doc_pages'] for q in pilot_questions])
mean_ans_page = np.mean([q['_actual_answer_page'] for q in pilot_questions])

print(f'=== Pilot Set Statistics (N={len(pilot_questions)}) ===')
print(f'  Unanswerable       : {n_unans} (0.0% -- MP-DocVQA is fully answerable)')
print(f'  Mean doc pages     : {mean_pgs:.1f}')
print(f'  Mean answer page   : {mean_ans_page:.1f}')
print(f'  Mean Q length (wds): {mean_q_words:.1f}')
print(f'  Mean A length (wds): {mean_a_words:.1f}')
print()
print(f'Top-5 doc_ids by question count:')
doc_q_count = Counter(q['doc_id'] for q in pilot_questions)
for did, cnt in doc_q_count.most_common(5):
    print(f'  {did}: {cnt} questions')

## Phase 1.5 -- Exploratory Data Analysis

Comprehensive descriptive statistics, charts, and tables for the full
MP-DocVQA val corpus and the pilot subset.  
All figures saved to `results/figures/`.

| Figure | Description |
|---|---|
| Fig 1 | Total pages per doc (histogram) + questions per unique doc (bar) |
| Fig 2 | Answer page (absolute) + normalised answer position distributions |
| Fig 3 | Question length + answer length distributions |
| Fig 4 | Context span width + context start page distributions |
| Fig 5 | Top-30 answer first words (answer type fingerprint) |
| Fig 6 | Answer consensus (unique valid answers per question) |
| Fig 7 | Pilot vs full val corpus key metrics comparison |

In [ ]:
import matplotlib
matplotlib.use('Agg')   # non-interactive backend for nbconvert
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
FIG_DIR = RESULTS_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True)

# --- Full corpus derived stats -------------------------------------------
N_ALL    = len(all_questions)
N_DOCS   = len({q['doc_id'] for q in all_questions})
MEAN_PGS = np.mean([q['total_doc_pages'] for q in all_questions])
MAX_PGS  = max(q['total_doc_pages'] for q in all_questions)
MEAN_QW  = np.mean([q['_q_words']    for q in all_questions])
MEAN_AW  = np.mean([q['_a_words']    for q in all_questions])
MEAN_ANS_PAGE = np.mean([q['_actual_answer_page'] for q in all_questions])

overview = pd.DataFrame({
    'Metric': [
        'Total QA records', 'Unique documents',
        'Unanswerable questions',
        'Mean doc pages',
        'Max doc pages',
        'Mean answer page (absolute)',
        'Mean question length (words)',
        'Mean answer length (words)',
    ],
    'Full val (5,187)': [
        f'{N_ALL:,}', f'{N_DOCS}',
        '0 (0.0%)',
        f'{MEAN_PGS:.1f}',
        f'{MAX_PGS}',
        f'{MEAN_ANS_PAGE:.1f}',
        f'{MEAN_QW:.1f}',
        f'{MEAN_AW:.1f}',
    ],
    f'Pilot (N={len(pilot_questions)})': [
        f'{len(pilot_questions)}', f'{len(pilot_doc_ids)}',
        '0 (0.0%)',
        f'{np.mean([q["total_doc_pages"] for q in pilot_questions]):.1f}',
        f'{max(q["total_doc_pages"] for q in pilot_questions)}',
        f'{np.mean([q["_actual_answer_page"] for q in pilot_questions]):.1f}',
        f'{np.mean([q["_q_words"] for q in pilot_questions]):.1f}',
        f'{np.mean([q["_a_words"] for q in pilot_questions]):.1f}',
    ],
})
print('=== Corpus Overview ===')
print(overview.to_string(index=False))

In [ ]:
# --- Figure 1: Doc pages distribution + questions per unique doc ----------
total_pages = [q['total_doc_pages'] for q in all_questions]
q_per_doc   = Counter(q['doc_id'] for q in all_questions)
qpd_vals    = list(q_per_doc.values())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MP-DocVQA -- Document Length & Question Load (full val, N=5,187)', fontsize=13)

# Total pages per doc (log-scale x)
axes[0].hist(total_pages, bins=40, color=sns.color_palette('muted')[0], edgecolor='white')
axes[0].set_xlabel('Total pages per document')
axes[0].set_ylabel('Number of questions')
axes[0].set_title(f'Document Length Distribution (mean={MEAN_PGS:.0f}, max={MAX_PGS})')
axes[0].axvline(np.mean(total_pages), color='red', linestyle='--', linewidth=1.2, label=f'Mean {np.mean(total_pages):.0f}')
axes[0].legend(fontsize=9)

# Questions per unique doc
qpd_counter = Counter(qpd_vals)
sorted_counts = sorted(qpd_counter.items())
x_vals = [k for k, _ in sorted_counts[:20]]
y_vals = [v for _, v in sorted_counts[:20]]
axes[1].bar(x_vals, y_vals, color=sns.color_palette('muted')[1], edgecolor='white')
axes[1].set_xlabel('Questions per document')
axes[1].set_ylabel('Number of documents')
axes[1].set_title(f'Questions per Document (N={N_DOCS} unique docs, top 20)')

plt.tight_layout()
fig.savefig(str(FIG_DIR / 'fig1_doc_length_dist.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR}/fig1_doc_length_dist.png')

In [ ]:
# --- Figure 2: Answer page distributions ---------------------------------
ans_pages     = [q['_actual_answer_page'] for q in all_questions]
norm_positions = [q['_actual_answer_page'] / max(q['total_doc_pages'] - 1, 1)
                  for q in all_questions]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
fig.suptitle('MP-DocVQA -- Answer Page Location (full val)', fontsize=13)

# Absolute answer page index
axes[0].hist(ans_pages, bins=50, color=sns.color_palette('muted')[2], edgecolor='white')
axes[0].set_xlabel('Absolute answer page index (0-based)')
axes[0].set_ylabel('Number of questions')
axes[0].set_title(f'Answer Page Distribution (mean={MEAN_ANS_PAGE:.1f})')
axes[0].axvline(np.mean(ans_pages), color='red', linestyle='--', linewidth=1.2,
                label=f'Mean {np.mean(ans_pages):.1f}')
axes[0].legend(fontsize=9)

# Normalised answer position [0, 1]
axes[1].hist(norm_positions, bins=50, color=sns.color_palette('muted')[3], edgecolor='white')
axes[1].set_xlabel('Normalised answer position (0 = first page, 1 = last page)')
axes[1].set_ylabel('Number of questions')
axes[1].set_title('Normalised Answer Position')
axes[1].axvline(np.mean(norm_positions), color='red', linestyle='--', linewidth=1.2,
                label=f'Mean {np.mean(norm_positions):.2f}')
axes[1].legend(fontsize=9)

plt.tight_layout()
fig.savefig(str(FIG_DIR / 'fig2_answer_page_dist.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR}/fig2_answer_page_dist.png')

In [ ]:
# --- Figure 3: Question and answer word length distributions -------------
q_words = [q['_q_words'] for q in all_questions]
a_words = [q['_a_words'] for q in all_questions]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('MP-DocVQA -- Question & Answer Length Distributions (full val)', fontsize=13)

axes[0].hist(q_words, bins=30, color=sns.color_palette('muted')[4], edgecolor='white')
axes[0].set_xlabel('Question length (words)')
axes[0].set_ylabel('Frequency')
axes[0].set_title(f'Question Length (mean={MEAN_QW:.1f} words)')
axes[0].axvline(np.mean(q_words), color='red', linestyle='--', linewidth=1.2)

axes[1].hist(a_words, bins=20, color=sns.color_palette('muted')[5], edgecolor='white')
axes[1].set_xlabel('Answer length (words)')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Answer Length (mean={MEAN_AW:.1f} words)')
axes[1].axvline(np.mean(a_words), color='red', linestyle='--', linewidth=1.2)

plt.tight_layout()
fig.savefig(str(FIG_DIR / 'fig3_qlen_alen_dist.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR}/fig3_qlen_alen_dist.png')

In [ ]:
# --- Figure 4: Context window span + start page distributions -----------
# pages field is the context window given in the dataset (1-2 pages)
span_widths  = [max(q['pages']) - min(q['pages']) + 1 for q in all_questions if q['pages']]
start_pages  = [min(q['pages']) for q in all_questions if q['pages']]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('MP-DocVQA -- Context Window Characteristics (full val)', fontsize=13)

# Span width (how many pages the dataset provides as context)
span_counts = Counter(span_widths)
sc_sorted   = sorted(span_counts.items())
axes[0].bar([k for k, _ in sc_sorted], [v for _, v in sc_sorted],
            color=sns.color_palette('muted')[0], edgecolor='white')
axes[0].set_xlabel('Context span (pages)')
axes[0].set_ylabel('Number of questions')
axes[0].set_title(f'Context Window Span Width (mean={np.mean(span_widths):.2f})')

# Context start page (how deep into the doc)
axes[1].hist(start_pages, bins=40, color=sns.color_palette('muted')[1], edgecolor='white')
axes[1].set_xlabel('Context start page (absolute)')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Context Start Page (mean={np.mean(start_pages):.1f})')
axes[1].axvline(np.mean(start_pages), color='red', linestyle='--', linewidth=1.2)

plt.tight_layout()
fig.savefig(str(FIG_DIR / 'fig4_context_window.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR}/fig4_context_window.png')

In [ ]:
# --- Figure 5: Top-30 answer first words (answer type fingerprint) -------
first_words = Counter()
for q in all_questions:
    ans = q['answers'][0] if q['answers'] else ''
    if ans:
        first_word = ans.strip().lower().split()[0] if ans.strip() else ''
        if first_word:
            first_words[first_word] += 1

top30 = first_words.most_common(30)
fw_labels = [w for w, _ in top30]
fw_vals   = [c for _, c in top30]

fig, ax = plt.subplots(figsize=(14, 6))
colors  = sns.color_palette('tab20', len(fw_labels))
bars    = ax.barh(fw_labels[::-1], fw_vals[::-1], color=colors)
ax.set_xlabel('Frequency')
ax.set_title('MP-DocVQA -- Top-30 Answer First Words (val corpus, N=5,187)', fontsize=13)
for bar, val in zip(bars, fw_vals[::-1]):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=8)

plt.tight_layout()
fig.savefig(str(FIG_DIR / 'fig5_answer_first_words.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR}/fig5_answer_first_words.png')

In [ ]:
# --- Figure 6: Answer consensus (unique valid answers per question) ------
# How many distinct valid answers annotators provided
n_unique_ans = [len(set(q['answers'])) for q in all_questions]
n_total_ans  = [len(q['answers']) for q in all_questions]

# Consensus score: 1.0 = all annotators agree, lower = more disagreement
from collections import Counter as _C
consensus_scores = []
for q in all_questions:
    if q['answers']:
        most_common_count = _C(q['answers']).most_common(1)[0][1]
        consensus_scores.append(most_common_count / len(q['answers']))
    else:
        consensus_scores.append(0.0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('MP-DocVQA -- Answer Annotation Agreement (full val)', fontsize=13)

unique_counter = Counter(n_unique_ans)
uc_sorted = sorted(unique_counter.items())
axes[0].bar([k for k, _ in uc_sorted], [v for _, v in uc_sorted],
            color=sns.color_palette('muted')[2], edgecolor='white')
axes[0].set_xlabel('Unique valid answers per question')
axes[0].set_ylabel('Number of questions')
axes[0].set_title('Answer Diversity (1 = all annotators agree)')

axes[1].hist(consensus_scores, bins=20, color=sns.color_palette('muted')[3], edgecolor='white')
axes[1].set_xlabel('Consensus score (majority vote / total annotations)')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Annotator Consensus (mean={np.mean(consensus_scores):.2f})')
axes[1].axvline(np.mean(consensus_scores), color='red', linestyle='--', linewidth=1.2)

plt.tight_layout()
fig.savefig(str(FIG_DIR / 'fig6_answer_consensus.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR}/fig6_answer_consensus.png')

In [ ]:
# --- Figure 7: Pilot vs full val comparison (5 key metrics) -------------
labels  = ['Total Qs', 'Unique docs', 'Mean doc pages', 'Mean Q len (wds)', 'Mean A len (wds)']
full_vals  = [N_ALL, N_DOCS, MEAN_PGS, MEAN_QW, MEAN_AW]
pilot_vals = [
    len(pilot_questions),
    len(pilot_doc_ids),
    np.mean([q['total_doc_pages'] for q in pilot_questions]),
    np.mean([q['_q_words'] for q in pilot_questions]),
    np.mean([q['_a_words'] for q in pilot_questions]),
]

x      = np.arange(len(labels))
width  = 0.35
norm_full  = [1.0] * len(labels)   # full = 100%
norm_pilot = [p / f if f else 0.0 for p, f in zip(pilot_vals, full_vals)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MP-DocVQA -- Pilot vs Full Val Corpus', fontsize=13)

# Raw values
axes[0].bar(x - width/2, full_vals,  width, label='Full val',       color=sns.color_palette('muted')[0])
axes[0].bar(x + width/2, pilot_vals, width, label=f'Pilot (N={N_PILOT})', color=sns.color_palette('muted')[1])
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=18, ha='right', fontsize=8)
axes[0].set_title('Raw Values')
axes[0].legend(fontsize=9)

# Pilot as fraction of full
axes[1].bar(x, norm_pilot, width=0.5, color=sns.color_palette('muted')[2])
axes[1].axhline(1.0, color='red', linestyle='--', linewidth=1, label='Full = 1.0')
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=18, ha='right', fontsize=8)
axes[1].set_title('Pilot as Fraction of Full Val')
axes[1].set_ylim(0, 1.3)
axes[1].legend(fontsize=9)

plt.tight_layout()
fig.savefig(str(FIG_DIR / 'fig7_pilot_vs_full.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR}/fig7_pilot_vs_full.png')

In [ ]:
# --- Table: Sample QA pairs (3 per answer type) -------------------------
# Categorise answers: numeric, short-text (1-2 words), longer-text (3+ words)
def answer_category(ans):
    try:
        float(ans.replace(',', '').replace('%', ''))
        return 'Numeric'
    except (ValueError, AttributeError):
        pass
    words = ans.strip().split()
    return 'Short-text (1-2 wds)' if len(words) <= 2 else 'Longer-text (3+ wds)'

print('=== Sample QA Pairs by Answer Category ===')
for cat in ['Numeric', 'Short-text (1-2 wds)', 'Longer-text (3+ wds)']:
    samples = [q for q in all_questions
               if q['answers'] and answer_category(q['answers'][0]) == cat][:3]
    if not samples:
        continue
    n_total = sum(1 for q in all_questions
                  if q['answers'] and answer_category(q['answers'][0]) == cat)
    print(f'\n--- {cat} ({n_total} questions) ---')
    for q in samples:
        ans_display = q['answers'][0][:80]
        print(f'  Q: {q["question"][:90]}')
        print(f'  A: {ans_display}')
        print(f'  doc_id={q["doc_id"]}  page={q["_actual_answer_page"]}  total_pages={q["total_doc_pages"]}')
        print()

## Phase 2 -- Verify OCR & Image File Coverage

MP-DocVQA data is already local -- no download required.  
This phase verifies that all pilot documents have their corresponding
OCR JSON files (`ocr/ocr/{doc_id}_p{N}.json`) and page images
(`images/images/{doc_id}_p{N}.jpg`) available.  

OCR location: `data/mpdocvqa/ocr/ocr/`  
Images location: `data/mpdocvqa/images/images/`

In [ ]:
# Resolve the actual OCR subdirectory (handles nested ocr/ocr/ layout)
actual_ocr_dir = OCR_DIR
nested_ocr = OCR_DIR / OCR_DIR.name
if nested_ocr.exists() and any(nested_ocr.glob('*.json')):
    actual_ocr_dir = nested_ocr

actual_img_dir = IMG_DIR
nested_img = IMG_DIR / IMG_DIR.name
if nested_img.exists() and any(nested_img.glob('*.jpg')):
    actual_img_dir = nested_img

print(f'OCR directory : {actual_ocr_dir}')
print(f'Image directory: {actual_img_dir}')

# Check OCR coverage for pilot docs
missing_ocr   = []
missing_img   = []
parseable_docs = []

for doc_id in pilot_doc_ids:
    ocr_files = list(actual_ocr_dir.glob(f'{doc_id}_p*.json'))
    img_files = list(actual_img_dir.glob(f'{doc_id}_p*.jpg'))
    if not ocr_files:
        missing_ocr.append(doc_id)
    if not img_files:
        missing_img.append(doc_id)
    if ocr_files:
        parseable_docs.append(doc_id)

print(f'\nPilot docs    : {len(pilot_doc_ids)}')
print(f'With OCR files: {len(parseable_docs)} ({len(parseable_docs)}/{len(pilot_doc_ids)})')
print(f'With images   : {len(pilot_doc_ids)-len(missing_img)} ({len(pilot_doc_ids)-len(missing_img)}/{len(pilot_doc_ids)})')
if missing_ocr:
    print(f'Warning: Missing OCR for: {missing_ocr[:5]}')
if missing_img:
    print(f'Warning: Missing images for: {missing_img[:5]}')

# Spot-check: count OCR pages for first 3 pilot docs
for doc_id in parseable_docs[:3]:
    n_ocr = len(list(actual_ocr_dir.glob(f'{doc_id}_p*.json')))
    n_img = len(list(actual_img_dir.glob(f'{doc_id}_p*.jpg')))
    print(f'  {doc_id}: {n_ocr} OCR pages, {n_img} images')

## Phase 3 -- Parse Documents with mpdocvqa_parser + Temporal Annotation

The `mpdocvqa_parser` converts AWS Textract OCR JSON files (one per page) into
the GraMM-RAG element format used by graph builder and embeddings.  
No MinerU required -- all data is already in text form.

Output: `parsed/mpdocvqa/{doc_id}.json` for each pilot document.  
**Runtime:** ~1-5 s/doc (pure JSON parsing, no GPU needed).

In [ ]:
from src.parsing.mpdocvqa_parser import parse_mpdocvqa_doc_from_pages

parsed_count = skip_count = fail_count = 0

for doc_id in parseable_docs:
    out_path  = PARSED_DIR / f'{doc_id}.json'
    if out_path.exists():
        skip_count += 1
        continue

    page_files = sorted(actual_ocr_dir.glob(f'{doc_id}_p*.json'),
                        key=lambda p: int(re.search(r'_p(\d+)', p.stem).group(1)))
    if not page_files:
        logger.warning(f'No OCR files for {doc_id}')
        fail_count += 1
        continue
    try:
        parsed = parse_mpdocvqa_doc_from_pages(page_files, doc_id)
        out_path.write_text(json.dumps(parsed, indent=2), encoding='utf-8')
        parsed_count += 1
    except Exception as e:
        logger.error(f'Parse failed {doc_id}: {e}')
        fail_count += 1

print(f'Parsed: {parsed_count}, Skipped (cached): {skip_count}, Failed: {fail_count}')
parsed_ids = {p.stem for p in PARSED_DIR.glob('*.json')}
print(f'Total parsed docs available: {len(parsed_ids)}')

In [ ]:
from src.parsing.temporal import annotate_parsed_elements

annotated = 0
for pf in sorted(PARSED_DIR.glob('*.json')):
    try:
        parsed = json.loads(pf.read_text(encoding='utf-8'))
        if any('temporal_markers' in e for e in parsed.get('elements', [])):
            continue
        parsed = annotate_parsed_elements(parsed, None, use_llm_fallback=False)
        pf.write_text(json.dumps(parsed, indent=2), encoding='utf-8')
        annotated += 1
    except Exception as e:
        logger.warning(f'Temporal annotation failed {pf.name}: {e}')

print(f'Temporal annotation: {annotated} docs annotated')

## Phase 4 -- Node Embeddings + PyG Graph Construction

**Text nodes** (all elements in MP-DocVQA are text type):  
E5-Mistral-7B-Instruct (4-bit, ~4 GB VRAM) -> 256-dim projection.  
Fallback: E5-large-v2 on CPU.

**No image nodes:** MP-DocVQA parsed elements are all `text` type (Textract LINE blocks).  
KG triplets: GPT-4o-mini extraction (skipped if `OPENAI_API_KEY` not set).  

**Runtime:** ~30-120 s/doc (embedding only, no PDF rendering).

In [ ]:
if SKIP_EMBEDDING:
    print('SKIP_EMBEDDING=True (pilot+CPU): skipping neural embedding step.')
    print('  Phases 5-9 will use random-init HGT graphs (no FAISS indices).')
    print('  Set SKIP_EMBEDDING=False with a GPU for full embedding.')
else:
    from src.graph.embeddings import compute_and_save_embeddings

    embedded_count = skip_emb_count = 0

    _pilot_set = set(pilot_doc_ids) if N_PILOT else None
    for pf in sorted(PARSED_DIR.glob('*.json')):
        doc_id   = pf.stem
        if _pilot_set is not None and doc_id not in _pilot_set:
            continue
        text_pt  = EMB_DIR / f'{doc_id}_text.pt'
        img_pt   = EMB_DIR / f'{doc_id}_img.pt'
        raw_pt   = EMB_DIR / f'{doc_id}_text_raw.pt'
        # Require the raw E5 file too: docs embedded before the Option-A
        # change lack it and must be recomputed for semantic retrieval.
        if text_pt.exists() and img_pt.exists() and raw_pt.exists():
            skip_emb_count += 1
            continue
        try:
            parsed = json.loads(pf.read_text(encoding='utf-8'))
            if MAX_PAGES_PILOT is not None:
                parsed = dict(parsed)
                parsed['elements'] = [e for e in parsed.get('elements', [])
                                       if e.get('page_no', 0) < MAX_PAGES_PILOT]
            compute_and_save_embeddings(
                parsed,
                embeddings_dir=str(EMB_DIR),
                image_root=str(actual_img_dir),
                device=DEVICE,
                text_batch_size=16,
                image_batch_size=8,
            )
            embedded_count += 1
        except Exception as e:
            logger.error(f'Embedding failed {doc_id}: {e}')

    print(f'Embeddings computed: {embedded_count}, Skipped (cached): {skip_emb_count}')

In [ ]:
from src.graph.edges import extract_all_triplets
from src.graph.builder import build_graph, save_graph

openai_client_kg = None
if os.environ.get('OPENAI_API_KEY'):
    try:
        import openai
        openai_client_kg = openai.OpenAI(api_key=os.environ['OPENAI_API_KEY'])
        print('OpenAI client ready for KG extraction')
    except ImportError:
        print('openai package not found -- KG triplets empty')
else:
    print('No OPENAI_API_KEY -- KG triplets empty (graph still built)')

EMBED_DIM  = 256   # projection dim used by DocumentHGT
graph_count = skip_graph = 0

_pilot_set = set(pilot_doc_ids) if N_PILOT else None
for pf in sorted(PARSED_DIR.glob('*.json')):
    doc_id  = pf.stem
    if _pilot_set is not None and doc_id not in _pilot_set:
        continue
    if (GRAPH_DIR / f'{doc_id}.pt').exists():
        skip_graph += 1
        continue
    # Graph text features = RAW E5 (un-projected). The HGT's trainable
    # node_lin (Linear(-1, 256)) becomes the projection and is learned in
    # Phase 5 -- replacing the old random/discarded projection so the
    # GRAPH/HYBRID path is semantic, not noise.
    text_pt = EMB_DIR / f'{doc_id}_text_raw.pt'
    img_pt  = EMB_DIR / f'{doc_id}_img.pt'
    try:
        parsed = json.loads(pf.read_text(encoding='utf-8'))
        parsed.setdefault('doc_id', doc_id)
        n_text = len([e for e in parsed.get('elements', []) if e['type'] == 'text'])
        if text_pt.exists():
            text_emb = torch.load(text_pt, weights_only=True)
        else:
            # Fallback (should not happen now SKIP_EMBEDDING=False): random init
            text_emb = torch.nn.init.xavier_uniform_(
                torch.empty(max(n_text, 1), EMBED_DIM))
        img_emb = (torch.load(img_pt, weights_only=True)
                   if img_pt.exists() else torch.zeros(0, EMBED_DIM))
        embeddings = {'text': text_emb, 'img': img_emb}
        triplets = []
        if openai_client_kg:
            try:
                triplets = extract_all_triplets(parsed, openai_client_kg)
            except Exception as e:
                logger.warning(f'KG extraction failed {doc_id}: {e}')
        graph = build_graph(doc_id, parsed, embeddings, triplets)
        save_graph(graph, str(GRAPH_DIR), doc_id)
        graph_count += 1
    except Exception as e:
        logger.error(f'Graph build failed {doc_id}: {e}')

emb_note = '(random-init)' if SKIP_EMBEDDING else '(neural)'
print(f'Graphs built {emb_note}: {graph_count}, Skipped (cached): {skip_graph}')
print(f'Total graphs available: {len(list(GRAPH_DIR.glob("*.pt")))}')

## Retrieval Helpers

`embed_query` embeds a question with the SAME E5 model/space the raw node
embeddings use (asymmetric instruction prefix). `node_pages` maps retrieved
nodes to their page numbers for APPA. Defined here so Phase 5 (HGT training
anchor), Phase 8 (vector) and Phase 9 (GraMM) all share one implementation.

In [ ]:
from src.graph.embeddings import load_text_model

_QUERY_MODEL = None
def embed_query(question: str):
    global _QUERY_MODEL
    if _QUERY_MODEL is None:
        _QUERY_MODEL, _ = load_text_model(use_4bit=(DEVICE == 'cuda'), device=DEVICE)
    q = f'Represent this question for retrieving relevant document passages: {question}'
    # convert_to_tensor=False -> numpy, then a FRESH torch tensor. Avoids
    # SentenceTransformer's inference-mode tensors, which cannot flow
    # through the trainable HGT node_lin during Phase 5 backprop.
    emb = _QUERY_MODEL.encode([q], normalize_embeddings=True)
    return torch.from_numpy(np.asarray(emb[0], dtype=np.float32))

def node_pages(nodes, parsed):
    """Map retrieved node dicts -> ordered list of their page_no (int),
    preserving retrieval rank order. Used for APPA."""
    if not parsed:
        return []
    type_to_elems = {}
    for e in parsed.get('elements', []):
        type_to_elems.setdefault(e['type'], []).append(e)
    pages = []
    for nd in nodes:
        nt, li = nd.get('node_type'), nd.get('local_idx')
        if nt is None or li is None:
            continue
        elems = type_to_elems.get(nt, [])
        if 0 <= li < len(elems):
            try:
                pages.append(int(elems[li].get('page_no', -1)))
            except (TypeError, ValueError):
                pass
    return pages

import torch.nn.functional as _Fnn
_RAWVEC_CACHE = {}
def _raw_node_vectors(doc_id, parsed):
    """Map (node_type, local_idx) -> raw-E5 vector for one doc, using the
    same contiguous text/section/equation slicing Phase 8a uses."""
    if doc_id in _RAWVEC_CACHE:
        return _RAWVEC_CACHE[doc_id]
    p = EMB_DIR / f'{doc_id}_text_raw.pt'
    out = {}
    if p.exists():
        te = torch.load(p, weights_only=True).float()
        from collections import Counter as _C
        tc = _C(e['type'] for e in parsed.get('elements', []))
        cur = 0
        for nt, cnt in [('text', tc.get('text', 0)),
                         ('section', tc.get('section', 0)),
                         ('equation', tc.get('equation', 0))]:
            if cnt > 0 and cur < te.shape[0]:
                end = min(cur + cnt, te.shape[0])
                for li in range(end - cur):
                    out[(nt, li)] = te[cur + li]
                cur = end
    _RAWVEC_CACHE[doc_id] = out
    return out

def rerank_by_raw_e5(nodes, q_vec, doc_id, parsed):
    """#1 Hybrid re-rank: self-healing/HGT owns the candidate set + reward;
    raw-E5 cosine to the question owns the ORDER the LLM sees. Converts
    GraMM's recall edge into generator-visible precision. Nodes without a
    raw vector keep their original score and sort last."""
    if not nodes:
        return nodes
    vmap = _raw_node_vectors(doc_id, parsed or {'elements': []})
    qn = _Fnn.normalize(q_vec.float().unsqueeze(0), dim=-1)
    scored = []
    for nd in nodes:
        v = vmap.get((nd.get('node_type'), nd.get('local_idx')))
        if v is not None:
            s = float(_Fnn.cosine_similarity(qn, v.unsqueeze(0)).item())
        else:
            s = -1.0 + float(nd.get('score', 0.0)) * 1e-3
        nd['rerank_score'] = s   # so page_context_nodes can aggregate it
        scored.append((s, nd))
    scored.sort(key=lambda t: t[0], reverse=True)
    return [nd for _, nd in scored]

def page_context_nodes(ranked_nodes, parsed, n_pages):
    """Page-aware context: rank pages by AGGREGATE retrieved-node score
    (PAGE_SCORE_MODE), take the top-`n_pages`, and return synthetic node
    dicts for ALL elements on those pages in reading order. build_prompt
    re-resolves text via (node_type, local_idx), so we emit those exactly.
    Aggregating (vs first-appearance) lets a page with many medium-scored
    nodes -- exactly what graph expansion produces on the answer page --
    win over a page with one spurious high node."""
    elements = (parsed or {}).get('elements', [])
    if not elements:
        return ranked_nodes
    # type_to_elems mirrors build_prompt: grouped by type, doc order.
    type_to_elems = {}
    for e in elements:
        type_to_elems.setdefault(e['type'], []).append(e)
    # Collect each page's retrieved-node scores (raw-E5 cosine if reranked,
    # else the path's own score). First-seen order breaks aggregate ties.
    page_scores, first_seen = {}, []
    for nd in ranked_nodes:
        nt, li = nd.get('node_type'), nd.get('local_idx')
        elems = type_to_elems.get(nt, [])
        if li is None or not (0 <= li < len(elems)):
            continue
        try:
            pg = int(elems[li].get('page_no', -1))
        except (TypeError, ValueError):
            continue
        sc = float(nd.get('rerank_score', nd.get('score', 0.0)))
        if pg not in page_scores:
            page_scores[pg] = []
            first_seen.append(pg)
        page_scores[pg].append(sc)
    if not page_scores:
        return ranked_nodes
    if PAGE_SCORE_MODE == 'first_seen':
        # Proven best: order pages by when their top-ranked node appeared.
        page_order = first_seen[:n_pages]
    else:
        def _agg(scs):
            if PAGE_SCORE_MODE == 'sum':
                return sum(scs)
            top = sorted(scs, reverse=True)[:max(1, PAGE_SCORE_TOPK)]
            return sum(top) / len(top)
        ranked_pages = sorted(first_seen,
                              key=lambda p: (-_agg(page_scores[p]),
                                             first_seen.index(p)))
        page_order = ranked_pages[:n_pages]
    sel = set(page_order)
    out = []
    for nt, elems in type_to_elems.items():
        for li, e in enumerate(elems):
            try:
                pg = int(e.get('page_no', -1))
            except (TypeError, ValueError):
                continue
            if pg in sel:
                try:
                    ro = int(e.get('reading_order', 0))
                except (TypeError, ValueError):
                    ro = 0
                # sort key: page rank, then reading order
                out.append((page_order.index(pg), ro,
                            {'node_type': nt, 'local_idx': li}))
    out.sort(key=lambda t: (t[0], t[1]))
    return [nd for _, _, nd in out]

_CE_MODEL = None
def cross_rerank(nodes, question, doc_id, parsed):
    """Two-stage retrieval precision step: jointly score (question, node
    text) with a cross-encoder. Bi-encoder cosine ranks close-but-wrong
    candidates (dates/labels) high; the cross-encoder catches them. Sets
    nd['rerank_score'] so page_context_nodes selects on cross scores."""
    if not nodes:
        return nodes
    global _CE_MODEL
    if _CE_MODEL is None:
        from sentence_transformers import CrossEncoder
        _CE_MODEL = CrossEncoder(CROSS_ENCODER_MODEL, device=DEVICE,
                                 max_length=512)
    elements = (parsed or {}).get('elements', [])
    type_to_elems = {}
    for e in elements:
        type_to_elems.setdefault(e['type'], []).append(e)
    pairs, idxs = [], []
    for j, nd in enumerate(nodes):
        nt, li = nd.get('node_type'), nd.get('local_idx')
        elems = type_to_elems.get(nt, [])
        txt = (elems[li].get('text', '') if li is not None
               and 0 <= li < len(elems) else nd.get('text', ''))
        if txt:
            pairs.append([question, txt[:2000]])
            idxs.append(j)
    if not pairs:
        return nodes
    scores = _CE_MODEL.predict(pairs, batch_size=32,
                               show_progress_bar=False)
    for j, sc in zip(idxs, scores):
        nodes[j]['rerank_score'] = float(sc)
    # Nodes with no resolvable text sort last.
    for j, nd in enumerate(nodes):
        nd.setdefault('rerank_score', -1e9)
    return sorted(nodes, key=lambda n: n['rerank_score'], reverse=True)

print('Retrieval helpers ready (embed_query, node_pages, '
      'rerank_by_raw_e5, page_context_nodes, cross_rerank).')

## Phase 5 -- Train HGT with Evidence-Guided InfoNCE Loss

Uses MP-DocVQA `answer_page_idx` + `pages` to identify the answer page as evidence:
- **Anchor:** the real question embedded via E5, projected through the HGT's
  trainable input layer (`encode_query`) -- question-conditioned, not a proxy
- **Positive:** text nodes on the answer page (`pages[answer_page_idx]`)
- **Negatives:** 15 random non-answer-page nodes

Saved to `results/models/hgt_mpdocvqa/best_model.pt`.

In [ ]:
from src.retrieval.hgt_model import DocumentHGT, info_nce_loss, METADATA
from src.graph.builder import load_graph

def get_evidence_node_indices_mpdocvqa(parsed, answer_page, node_type='text'):
    """Return indices of elements on the answer page for a given node type."""
    typed = [e for e in parsed['elements'] if e['type'] == node_type]
    return [i for i, e in enumerate(typed) if e.get('page_no', -1) == answer_page]

def build_hgt_triple_mpdocvqa(item, model, GRAPH_DIR, PARSED_DIR, device):
    doc_id      = item['doc_id']
    answer_page = item['_actual_answer_page']
    graph = load_graph(str(GRAPH_DIR), doc_id)
    if graph is None:
        return None
    pf = PARSED_DIR / f'{doc_id}.json'
    if not pf.exists():
        return None
    parsed = json.loads(pf.read_text(encoding='utf-8'))

    x_dict = {nt: graph[nt].x.to(device)
              for nt in graph.node_types
              if hasattr(graph[nt], 'x') and graph[nt].x.shape[0] > 0}
    edge_index_dict = {et: graph[et].edge_index.to(device)
                       for et in graph.edge_types
                       if graph[et].edge_index.shape[1] > 0}
    if not x_dict:
        return None

    out_dict = model(x_dict, edge_index_dict)
    text_out = out_dict.get('text')
    if text_out is None or text_out.shape[0] == 0:
        return None
    # Anchor = REAL question, embedded via E5 then projected through the
    # HGT's trainable input layer. node_lin is initialised by the model()
    # call above, so encode_query can use it. This trains the model to
    # align question space with answer-page nodes (was: doc-mean proxy).
    q_raw = embed_query(item['question']).to(device)
    q_emb = model.encode_query(q_raw)

    # Positive: nodes on the answer page
    pos_idxs = get_evidence_node_indices_mpdocvqa(parsed, answer_page, 'text')
    pos_embs = [torch.nn.functional.normalize(text_out[i], dim=-1)
                for i in pos_idxs if i < text_out.shape[0]]

    all_embs = torch.cat([e for e in out_dict.values() if e.shape[0] > 0])
    if all_embs.shape[0] < 2:
        return None
    pos_emb = (random.choice(pos_embs) if pos_embs
               else torch.nn.functional.normalize(
                    all_embs[random.randint(0, all_embs.shape[0]-1)], dim=-1))

    n_neg    = min(15, all_embs.shape[0] - 1)
    perm     = torch.randperm(all_embs.shape[0])[:n_neg]
    neg_embs = torch.nn.functional.normalize(all_embs[perm], dim=-1)
    return q_emb, pos_emb, neg_embs

print('HGT training helpers defined.')

In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)

available_graph_ids = [p.stem for p in GRAPH_DIR.glob('*.pt')]
graph_id_set        = set(available_graph_ids)

doc_to_questions = defaultdict(list)
for q in pilot_questions:
    if q['doc_id'] in graph_id_set:
        doc_to_questions[q['doc_id']].append(q)

print(f'HGT_EPOCHS = {HGT_EPOCHS}')
print(f'Training graphs available: {len(available_graph_ids)}')
print(f'Docs with questions: {len(doc_to_questions)}')

if hgt_save_path.exists():
    print(f'HGT already trained: {hgt_save_path}')
elif not available_graph_ids:
    print('No graphs available. Run Phase 4 first to build graphs.')
else:
    hgt_model = DocumentHGT(metadata=METADATA).to(DEVICE)
    optimizer = torch.optim.Adam(hgt_model.parameters(), lr=1e-3, weight_decay=1e-4)
    best_loss = float('inf')

    for epoch in range(HGT_EPOCHS):
        hgt_model.train()
        total_loss = n_batches = 0
        all_qs = [q for qs in doc_to_questions.values() for q in qs]
        random.shuffle(all_qs)
        for item in all_qs:
            triple = build_hgt_triple_mpdocvqa(item, hgt_model, GRAPH_DIR, PARSED_DIR, DEVICE)
            if triple is None:
                continue
            q_emb, pos_emb, neg_embs = triple
            optimizer.zero_grad()
            loss = info_nce_loss(q_emb, pos_emb, neg_embs)
            if not (torch.isnan(loss) or torch.isinf(loss)):
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
                n_batches  += 1
        avg_loss = total_loss / max(n_batches, 1)
        print(f'Epoch {epoch+1:3d}/{HGT_EPOCHS}  loss={avg_loss:.4f}  batches={n_batches}')
        if avg_loss < best_loss:
            best_loss = avg_loss
            hgt_save_path.parent.mkdir(parents=True, exist_ok=True)
            torch.save(hgt_model.state_dict(), hgt_save_path)

    print(f'HGT training done. Best loss={best_loss:.4f}  Saved: {hgt_save_path}')

## Phase 6 -- Train Query Router (DeBERTa-v3-base)

Auto-annotate routing labels from MP-DocVQA metadata:
- **GRAPH** -- answer is deep in doc (actual answer page > 5) -- cross-page navigation needed
- **VECTOR** -- answer is on early pages (answer page <= 5) -- simple lookup may suffice
- **HYBRID** -- not used (MP-DocVQA is text-only, no Chart/Figure/Table evidence)

Pilot mode skips DeBERTa fine-tuning (too slow on CPU).  
Phase 9 uses the heuristic regex router as fallback.  
Full run (`N_PILOT=None`): fine-tunes on all 5,187 val questions.  
Saved to `results/models/router_mpdocvqa/best_model/`.

In [ ]:
from src.retrieval.router import LABEL2ID

# Build router samples: GRAPH if answer is deep in document, else VECTOR
# (MP-DocVQA has no visual elements so HYBRID is not used)
def make_router_samples(questions):
    samples = []
    for q in questions:
        ans_page = q['_actual_answer_page']
        total_p  = q['total_doc_pages']
        # Heuristic: answer beyond page 5 in a long doc = GRAPH routing
        label = 'GRAPH' if (ans_page > 5 and total_p > 10) else 'VECTOR'
        samples.append({'query': q['question'], 'label': label})
    return samples

samples = make_router_samples(all_questions)
label_counts = Counter(s['label'] for s in samples)
print(f'Router training samples: {len(samples)}')
print(f'Label distribution: {dict(label_counts)}')
print(f'(HYBRID=0: MP-DocVQA is text-only, no visual routing needed)')

In [ ]:
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)

# -- Pilot mode: skip DeBERTa fine-tuning (CPU training >10 min per epoch)
# -- Phase 9 will use the heuristic router fallback (regex-based routing)
# -- Full run (N_PILOT=None): fine-tune DeBERTa for ROUTER_EPOCHS epochs

if N_PILOT:
    print('Pilot mode: skipping DeBERTa fine-tuning.')
    print('  Phase 9 will use the heuristic router (regex-based).')
    print('  Set N_PILOT=None for full DeBERTa fine-tuning (3 epochs).')
elif router_save_dir.exists() and any(router_save_dir.iterdir()):
    print(f'Router already trained: {router_save_dir}')
else:
    print(f'ROUTER_EPOCHS = {ROUTER_EPOCHS}')
    id2label = {0: 'GRAPH', 1: 'VECTOR', 2: 'HYBRID'}
    label2id = {v: k for k, v in id2label.items()}
    ds = Dataset.from_dict({
        'text':  [s['query'] for s in samples],
        'label': [label2id[s['label']] for s in samples],
    })
    split    = ds.train_test_split(test_size=0.15, seed=SEED)
    train_ds = split['train']
    val_ds   = split['test']

    model_name = 'microsoft/deberta-v3-base'
    tokenizer  = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch):
        return tokenizer(batch['text'], truncation=True,
                         max_length=128, padding='max_length')

    train_enc = train_ds.map(tokenize, batched=True)
    val_enc   = val_ds.map(tokenize,   batched=True)

    router_model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=3,
        id2label=id2label, label2id=label2id,
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {'accuracy': float((preds == labels).mean())}

    training_args = TrainingArguments(
        output_dir=str(MODEL_DIR / 'router_mpdocvqa'),
        num_train_epochs=ROUTER_EPOCHS,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='accuracy',
        logging_steps=50,
        seed=SEED,
        report_to='none',
    )
    trainer = Trainer(
        model=router_model, args=training_args,
        train_dataset=train_enc, eval_dataset=val_enc,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    trainer.save_model(str(router_save_dir))
    tokenizer.save_pretrained(str(router_save_dir))
    res = trainer.evaluate()
    print(f'Router val accuracy: {res.get("eval_accuracy", 0):.4f}')
    print(f'Saved: {router_save_dir}')

## Phase 7 -- Tune Reward Function (alpha, beta, lambda, tau Grid Search)

`R(q,K) = alpha*C + beta*S + lambda*T` -- C=coverage, S=similarity, T=temporal coherence

Grid search on a 20% holdout of pilot questions.  
Tuning signal: Spearman correlation between reward score and answer-page hit rate.

In [ ]:
from src.retrieval.hgt_model import retrieve_top_k
from src.retrieval.self_healing import RewardFunction
import itertools

hgt_model_tune = DocumentHGT(metadata=METADATA).to(DEVICE)
if hgt_save_path.exists():
    hgt_model_tune.load_state_dict(
        torch.load(hgt_save_path, map_location=DEVICE, weights_only=False))
    print(f'HGT loaded: {hgt_save_path}')
else:
    print('HGT not trained yet -- using random-init weights for reward tuning')
hgt_model_tune.eval()

random.seed(SEED + 1)
tuning_qs = random.sample(pilot_questions, max(4, int(0.2 * len(pilot_questions))))
print(f'Reward tuning on {len(tuning_qs)} held-out questions')

grid = [(a, b, l, t)
        for a, b, l, t in itertools.product(
            [0.30, 0.40, 0.50], [0.30, 0.35, 0.40],
            [0.20, 0.25, 0.30], [0.40, 0.50, 0.60, 0.70])
        if abs(a + b + l - 1.0) <= 0.05]
print(f'Grid size (weight-valid): {len(grid)} configurations')

if reward_save.exists():
    print(f'Reward params already tuned: {reward_save}')
else:
    best_score, best_params = -1.0, None
    for alpha, beta, lambda_, tau in grid:
        rf = RewardFunction(alpha=alpha, beta=beta, lambda_=lambda_, tau=tau)
        scores, hits = [], []
        for item in tuning_qs:
            doc_id = item['doc_id']
            graph  = load_graph(str(GRAPH_DIR), doc_id)
            pf     = PARSED_DIR / f'{doc_id}.json'
            if graph is None or not pf.exists():
                continue
            parsed = json.loads(pf.read_text(encoding='utf-8'))
            with torch.no_grad():
                node_emb_dict = hgt_model_tune.encode_nodes(
                    {nt: graph[nt].x.to(DEVICE) for nt in graph.node_types
                     if hasattr(graph[nt], 'x') and graph[nt].x.shape[0] > 0},
                    {et: graph[et].edge_index.to(DEVICE) for et in graph.edge_types
                     if graph[et].edge_index.shape[1] > 0},
                )
            text_pt = EMB_DIR / f'{doc_id}_text.pt'
            q_emb = (torch.load(text_pt, weights_only=True).mean(0)
                     if text_pt.exists() else torch.zeros(256))
            nodes = retrieve_top_k(q_emb, node_emb_dict, top_k=10)
            rd    = rf.compute(q_emb, nodes, node_emb_dict, parsed, [])
            scores.append(rd['reward'])
            # Hit: did retrieval find any node on the answer page?
            ev_page = item['_actual_answer_page']
            ret_pages = set()
            for n in nodes:
                typed = [e for e in parsed['elements'] if e['type'] == n['node_type']]
                if n['local_idx'] < len(typed):
                    ret_pages.add(typed[n['local_idx']].get('page_no', -1))
            hits.append(1.0 if ev_page in ret_pages else 0.0)
        if len(scores) > 2 and sum(hits) > 0:
            from scipy.stats import spearmanr
            corr, _ = spearmanr(scores, hits)
            corr = 0.0 if corr != corr else float(corr)
        else:
            corr = 0.0
        if corr > best_score:
            best_score = corr
            best_params = {'alpha': alpha, 'beta': beta, 'lambda_': lambda_, 'tau': tau}

    if best_params is None:
        best_params = {'alpha': 0.40, 'beta': 0.35, 'lambda_': 0.25, 'tau': 0.60}
        print('Grid search inconclusive. Using defaults.')
    reward_save.parent.mkdir(parents=True, exist_ok=True)
    reward_save.write_text(json.dumps(best_params, indent=2), encoding='utf-8')
    print(f'Best reward params (corr={best_score:.3f}): {best_params}')

print('Reward tuning complete.')

## Phase 8 -- Flat-Vector RAG Baseline

Baseline: FAISS vector search (no graph, no self-healing) + LLM.  
Uses the same VLM backend as GraMM-RAG to isolate retrieval contribution.  
Gold: `valid_answers` list (ANLS computed as max over all valid answers).  

Results saved to `results/baseline_vector_mpdocvqa_pilot.json`.

In [ ]:
from src.retrieval.vector_retrieval import VectorIndex
from src.generation.prompt_builder import build_prompt
from src.generation.vlm_client import VLMClient
from src.evaluation.metrics import compute_all_metrics
# embed_query / node_pages defined earlier (Retrieval Helpers cell).

faiss_indices = {}

# Build the FAISS index IN MEMORY from the cached raw-E5 embeddings every
# run. We deliberately do NOT persist/reload a .index file: building
# IndexFlatIP over a few hundred vectors is sub-millisecond, and a
# persisted index is the classic stale-cache trap (a file from an earlier
# broken run gets reloaded and silently returns garbage). _text_raw.pt is
# the real (expensive) cache; the index is always rebuilt from it.
_pilot_set = set(pilot_doc_ids) if N_PILOT else None
for pf in sorted(PARSED_DIR.glob('*.json')):
    doc_id  = pf.stem
    if _pilot_set is not None and doc_id not in _pilot_set:
        continue
    text_pt = EMB_DIR / f'{doc_id}_text_raw.pt'
    if not text_pt.exists():
        continue
    try:
        parsed   = json.loads(pf.read_text(encoding='utf-8'))
        text_emb = torch.load(text_pt, weights_only=True).float()
        elements = parsed.get('elements', [])
        tc       = Counter(e['type'] for e in elements)
        node_emb_dict = {}
        cur = 0
        for nt, cnt in [('text', tc.get('text', 0)),
                         ('section', tc.get('section', 0)),
                         ('equation', tc.get('equation', 0))]:
            if cnt > 0 and cur < text_emb.shape[0]:
                end = min(cur + cnt, text_emb.shape[0])
                node_emb_dict[nt] = text_emb[cur:end]
                cur = end
        if node_emb_dict:
            vi = VectorIndex()
            vi.build(node_emb_dict, parsed, doc_id)
            if vi.index is not None:
                faiss_indices[doc_id] = vi
    except Exception as e:
        logger.warning(f'FAISS build failed {doc_id}: {e}')

print(f'FAISS indices built (in-memory, from _text_raw.pt): {len(faiss_indices)}')

In [ ]:
if vector_out_path.exists():
    print(f'Flat-vector RAG results exist: {vector_out_path}')
    vector_results = json.loads(vector_out_path.read_text())
else:
    vlm_vec = VLMClient(provider='together')
    v_preds, v_golds, v_refused = [], [], []
    v_ret_pages, v_gold_pages = [], []

    for i, item in enumerate(pilot_questions):
        question = item['question']
        gold     = item['answers']   # list of valid answers for ANLS
        doc_id   = item['doc_id']

        q_emb = embed_query(question)

        vi    = faiss_indices.get(doc_id)
        _pool = RERANK_POOL if USE_CROSS_ENCODER else 10
        nodes = vi.search(q_emb, top_k=_pool) if vi and vi.index else []

        pf     = PARSED_DIR / f'{doc_id}.json'
        parsed = json.loads(pf.read_text(encoding='utf-8')) if pf.exists() else {'elements': []}

        # Cross-encoder precision rerank (or bi-encoder sort). APPA below
        # still uses the bi-encoder retrieval order (node_pages(nodes)).
        if USE_CROSS_ENCODER:
            ranked = cross_rerank(list(nodes), question, doc_id, parsed)
        else:
            ranked = sorted(nodes, key=lambda n: n.get('score', 0.0),
                            reverse=True)
        if PAGE_CONTEXT:
            gen_nodes = page_context_nodes(ranked, parsed, PAGE_CONTEXT_N)
            mcc = PAGE_CONTEXT_CHARS
        else:
            gen_nodes = ranked[:GEN_TOP_N]
            mcc = 4000
        prompt = build_prompt(question, gen_nodes, parsed, {}, False,
                              benchmark='mpdocvqa', max_context_chars=mcc)
        answer = vlm_vec.generate(prompt, temperature=0.0, max_tokens=128)

        v_preds.append(answer)
        v_golds.append(gold)     # list for multi-annotator ANLS
        v_refused.append(False)
        v_ret_pages.append(node_pages(nodes, parsed))
        v_gold_pages.append(int(item.get('_actual_answer_page', -1)))

        if (i + 1) % 10 == 0:
            print(f'  Vector RAG [{i+1}/{len(pilot_questions)}]')

    v_metrics = compute_all_metrics(v_preds, v_golds,
                                    [True] * len(v_preds), v_refused,
                                    retrieved_pages=v_ret_pages,
                                    gold_pages=v_gold_pages)
    vector_results = {
        'system': 'Flat-vector RAG', 'n_questions': len(pilot_questions),
        'metrics': v_metrics, 'predictions': v_preds, 'golds': v_golds,
        'retrieved_pages': v_ret_pages, 'gold_pages': v_gold_pages,
    }
    vector_out_path.write_text(json.dumps(vector_results, indent=2), encoding='utf-8')
    print(f'Flat-vector RAG: ANLS={v_metrics["anls"]:.3f} '
          f'F1={v_metrics["f1"]:.3f} Acc={v_metrics["accuracy"]:.3f} '
          f'APPA@1={v_metrics["appa@1"]:.3f} APPA@10={v_metrics["appa@10"]:.3f}')
    print(f'Saved: {vector_out_path}')

## Phase 9 -- GraMM-RAG Full Evaluation (Single Seed)

Full pipeline: **route -> retrieve (HGT/FAISS) -> self-heal -> generate -> score**

Single seed matches the flat-vector baseline's protocol (1 pass) and the
deterministic inference config (temperature=0, fixed weights). Statistical
rigor comes from paired bootstrap (1,000 resamples) in the analysis notebook,
not seed averaging.

Gold answers: `valid_answers` list (ANLS computed as max over all valid answers).  
Results saved to `results/gramm_mpdocvqa_pilot_s{seed}.json`.

In [ ]:
from src.retrieval.router import QueryRouter
from src.retrieval.self_healing import SelfHealingRetriever

hgt_model_eval = DocumentHGT(metadata=METADATA).to(DEVICE)
if hgt_save_path.exists():
    hgt_model_eval.load_state_dict(
        torch.load(hgt_save_path, map_location=DEVICE, weights_only=False))
    print(f'HGT loaded: {hgt_save_path}')
else:
    print('Warning: HGT weights not found -- using random init')
hgt_model_eval.eval()

router = QueryRouter(
    model_dir=str(router_save_dir) if router_save_dir.exists() else None,
    device=DEVICE,
)
print(f'Router: {"DeBERTa" if router.model else "heuristic fallback"}')

reward_fn = RewardFunction(
    params_path=str(reward_save) if reward_save.exists() else None)

retriever = SelfHealingRetriever(
    hgt_model=hgt_model_eval, reward_fn=reward_fn,
    top_k=10, expansion_k=5, max_rounds=2,
)
vlm_eval = VLMClient(provider='together')
print('All GraMM-RAG components loaded.')

In [ ]:
def run_gramm_rag_mpdocvqa(questions, seed):
    random.seed(seed)
    torch.manual_seed(seed)
    predictions, golds, refused_flags, reward_scores = [], [], [], []
    ret_pages, gold_pages, routes = [], [], []

    for i, item in enumerate(questions):
        question   = item['question']
        gold_list  = item['answers']   # list of valid answers
        doc_id     = item['doc_id']

        graph  = load_graph(str(GRAPH_DIR), doc_id)
        pf     = PARSED_DIR / f'{doc_id}.json'
        parsed = json.loads(pf.read_text(encoding='utf-8')) if pf.exists() else None
        route  = router.route(question)

        # Single question-conditioned raw-E5 query for BOTH paths.
        # GRAPH/HYBRID: SelfHealingRetriever projects it through the HGT's
        # trained node_lin via encode_query (same space as Phase 5 training).
        q_vec = embed_query(question)

        if graph and parsed and route in ('GRAPH', 'HYBRID'):
            result = retriever.retrieve(q_vec, graph, parsed, [])
        else:
            vi    = faiss_indices.get(doc_id)
            nodes = vi.search(q_vec, top_k=10) if vi and vi.index else []
            result = {'nodes': nodes, 'reward': 0.5, 'refused': False,
                      'rounds': 0, 'reward_detail': {}}

        # Two-stage: self-healing chose the candidate set + reward; the
        # cross-encoder (or bi-encoder fallback) decides the precision
        # order. Then page-aware context. APPA below still uses the full
        # retrieved set (node_pages(result['nodes'])), unaffected.
        if USE_CROSS_ENCODER:
            ranked = cross_rerank(list(result['nodes']), question,
                                  doc_id, parsed)
        else:
            ranked = rerank_by_raw_e5(result['nodes'], q_vec, doc_id, parsed)
        if PAGE_CONTEXT:
            gen_nodes = page_context_nodes(ranked, parsed, PAGE_CONTEXT_N)
            mcc = PAGE_CONTEXT_CHARS
        else:
            gen_nodes = ranked[:GEN_TOP_N]
            mcc = 4000
        prompt = build_prompt(
            question, gen_nodes,
            parsed or {'elements': []},
            result['reward_detail'], result['refused'],
            benchmark='mpdocvqa', max_context_chars=mcc,
        )
        answer = vlm_eval.generate(prompt, temperature=0.0, max_tokens=128)

        predictions.append(answer)
        golds.append(gold_list)
        refused_flags.append(result['refused'])
        reward_scores.append(result['reward'])
        ret_pages.append(node_pages(result['nodes'], parsed))
        gold_pages.append(int(item.get('_actual_answer_page', -1)))
        routes.append(route)

        if (i + 1) % 10 == 0:
            print(f'  [{i+1}/{len(questions)}] route={route} '
                  f'R={result["reward"]:.2f} refused={result["refused"]}')

    # MP-DocVQA: all questions are answerable
    is_answerable = [True] * len(predictions)
    metrics = compute_all_metrics(predictions, golds, is_answerable,
                                  refused_flags,
                                  retrieved_pages=ret_pages,
                                  gold_pages=gold_pages)
    return {
        'n_questions':   len(questions),
        'metrics':       metrics,
        'predictions':   predictions,
        'golds':         golds,
        'reward_scores': reward_scores,
        'refused_flags': refused_flags,
        'retrieved_pages': ret_pages,
        'gold_pages':    gold_pages,
        'routes':        routes,
    }

print('run_gramm_rag_mpdocvqa() defined.')

In [ ]:
gramm_results = {}

for seed in [SEED]:
    out_path = RESULTS_DIR / f'gramm_mpdocvqa_pilot_s{seed}.json'
    if out_path.exists():
        print(f'Seed {seed}: results exist at {out_path}')
        gramm_results[seed] = json.loads(out_path.read_text())
        continue
    print(f'\n=== GraMM-RAG / seed={seed} ===')
    res = run_gramm_rag_mpdocvqa(pilot_questions, seed=seed)
    gramm_results[seed] = res
    out_path.write_text(json.dumps(res, indent=2), encoding='utf-8')
    m = res['metrics']
    print(f'  ANLS={m["anls"]:.3f}  F1={m["f1"]:.3f}  Acc={m["accuracy"]:.3f}  '
          f'APPA@1={m["appa@1"]:.3f}  APPA@10={m["appa@10"]:.3f}  '
          f'Refused={sum(res["refused_flags"])}/{res["n_questions"]}')

print('\nAll seeds complete.')

## Phase 10 -- Results Comparison Table

Compare GraMM-RAG against the flat-vector RAG baseline (same single-seed,
deterministic protocol).  
Primary metric: ANLS (DocVQA standard).

In [ ]:
import pandas as pd

rows = []

if vector_out_path.exists():
    vr = json.loads(vector_out_path.read_text())
    m  = vr['metrics']
    rows.append({
        'System': 'Flat-vector RAG',
        'N': vr['n_questions'],
        'ANLS': f"{m['anls']:.3f}",
        'F1':   f"{m['f1']:.3f}",
        'Accuracy': f"{m['accuracy']:.3f}",
        'APPA@1':  f"{m.get('appa@1', float('nan')):.3f}",
        'APPA@5':  f"{m.get('appa@5', float('nan')):.3f}",
        'APPA@10': f"{m.get('appa@10', float('nan')):.3f}",
        'Refused': 0, 'Seed': '-',
    })

for seed, res in sorted(gramm_results.items()):
    m = res['metrics']
    rows.append({
        'System': 'GraMM-RAG',
        'N': res['n_questions'],
        'ANLS': f"{m['anls']:.3f}",
        'F1':   f"{m['f1']:.3f}",
        'Accuracy': f"{m['accuracy']:.3f}",
        'APPA@1':  f"{m.get('appa@1', float('nan')):.3f}",
        'APPA@5':  f"{m.get('appa@5', float('nan')):.3f}",
        'APPA@10': f"{m.get('appa@10', float('nan')):.3f}",
        'Refused': sum(res['refused_flags']),
        'Seed': seed,
    })

if rows:
    df = pd.DataFrame(rows)
    print('=== GraMM-RAG vs Flat-Vector RAG on MP-DocVQA ===')
    print(f'Pilot: N_PILOT={N_PILOT} questions, Seed={SEED}')
    print()
    print(df.to_string(index=False))
else:
    print('No results yet. Run Phases 8-9 first.')

### Per-Route Breakdown (#2 instrumentation)

GraMM's value is only exercised on GRAPH/HYBRID-routed questions -- VECTOR-
routed ones use the same FAISS+query as the baseline by construction. This
shows the route distribution and per-route ANLS / APPA@10, so we can see
*where* GraMM wins or loses instead of inferring it from the aggregate.

In [ ]:
from src.evaluation.metrics import compute_anls, compute_answer_page_accuracy

for seed, res in sorted(gramm_results.items()):
    rts = res.get('routes')
    if not rts:
        print(f'seed {seed}: no route log (re-run Phase 9 with instrumentation)')
        continue
    P, G = res['predictions'], res['golds']
    RP, GP = res.get('retrieved_pages', []), res.get('gold_pages', [])
    from collections import Counter as _C
    dist = _C(rts)
    print(f'\n=== GraMM-RAG seed {seed} -- route distribution: '
          f'{dict(dist)} (of {len(rts)}) ===')
    print(f'{"Route":<10}{"N":>5}{"ANLS":>9}{"APPA@10":>9}')
    for rt in ['VECTOR', 'GRAPH', 'HYBRID']:
        idx = [i for i, r in enumerate(rts) if r == rt]
        if not idx:
            continue
        sp = [P[i] for i in idx]; sg = [G[i] for i in idx]
        a = compute_anls(sp, sg)
        if RP and GP:
            ap = compute_answer_page_accuracy(
                [RP[i] for i in idx], [GP[i] for i in idx]).get('appa@10', float('nan'))
        else:
            ap = float('nan')
        print(f'{rt:<10}{len(idx):>5}{a:>9.3f}{ap:>9.3f}')
    # Aggregate for reference
    print(f'{"ALL":<10}{len(rts):>5}{compute_anls(P, G):>9.3f}'
          f'{res["metrics"].get("appa@10", float("nan")):>9.3f}')

### APPA Analysis -- Answer-Page Prediction Accuracy

Left: APPA@k (k=1,5,10) -- fraction of questions where retrieval surfaced
the gold answer page within the top-k nodes. Right: rank at which the gold
answer page first appears in the retrieved list (lower = better localisation;
'miss' = gold page never retrieved). This isolates retrieval quality from
generation quality.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

FIG_DIR = RESULTS_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

def first_hit_rank(ret_pages, gold_pages):
    """Per-question rank (0-based) where gold page first appears; None if miss."""
    ranks = []
    for pages, gold in zip(ret_pages, gold_pages):
        gold_set = set(gold) if isinstance(gold, (list, set, tuple)) else {gold}
        r = next((j for j, p in enumerate(pages) if p in gold_set), None)
        ranks.append(r)
    return ranks

panels = []
if vector_out_path.exists():
    vr = json.loads(vector_out_path.read_text())
    panels.append(('Flat-vector RAG', vr['metrics'],
                   vr.get('retrieved_pages'), vr.get('gold_pages')))
for seed, res in sorted(gramm_results.items()):
    panels.append((f'GraMM-RAG (s{seed})', res['metrics'],
                   res.get('retrieved_pages'), res.get('gold_pages')))

ks = [1, 5, 10]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

width = 0.8 / max(len(panels), 1)
x = np.arange(len(ks))
for idx, (name, m, _rp, _gp) in enumerate(panels):
    vals = [m.get(f'appa@{k}', 0.0) for k in ks]
    ax1.bar(x + idx * width, vals, width, label=name)
ax1.set_xticks(x + width * (len(panels) - 1) / 2)
ax1.set_xticklabels([f'APPA@{k}' for k in ks])
ax1.set_ylabel('Accuracy'); ax1.set_ylim(0, 1)
ax1.set_title('Answer-Page Prediction Accuracy'); ax1.legend(fontsize=8)
ax1.grid(axis='y', alpha=0.3)

for name, _m, rp, gp in panels:
    if not rp or not gp:
        continue
    ranks = first_hit_rank(rp, gp)
    hit = [r for r in ranks if r is not None]
    miss = sum(1 for r in ranks if r is None)
    ax2.hist(hit, bins=range(0, 12), alpha=0.5,
             label=f'{name} (miss={miss}/{len(ranks)})')
ax2.set_xlabel('Rank of gold answer page in retrieved list (0 = top)')
ax2.set_ylabel('# questions')
ax2.set_title('Gold-Page Retrieval Rank'); ax2.legend(fontsize=8)
ax2.grid(axis='y', alpha=0.3)

fig.tight_layout()
fig.savefig(str(FIG_DIR / 'fig_appa_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR}/fig_appa_comparison.png')

In [ ]:
summary = {
    'N_PILOT': N_PILOT, 'SEED': SEED, 'benchmark': 'MP-DocVQA',
    'flat_vector_rag': json.loads(vector_out_path.read_text())['metrics']
        if vector_out_path.exists() else None,
    'gramm_rag': {str(s): r['metrics'] for s, r in gramm_results.items()},
}
(RESULTS_DIR / 'summary_mpdocvqa_pilot.json').write_text(
    json.dumps(summary, indent=2), encoding='utf-8')

print('All results saved to results/')
print()
print('To extend to the FULL 5,187-question run:')
print('  1. Set N_PILOT = None  in Phase 0 (Cell 1)')
print('  2. Re-run all cells')
print('  3. Estimated API cost: ~$8-15 (Llama-3.3-70B-Turbo, single seed, 5,187 Qs)')